# FDA label extraction: QLoRA fine-tune of Qwen2.5-1.5B-Instruct (Phase 4)
Settings (Accelerator = **GPU T4 x2**, Internet = **On**) are in the README / PLAN.md (M4). Everything is fixed in advance: 1,200 training labels, 1 epoch, fp16, LoRA r=16. Run the cells in order. The adapter ends up in the Output tab as `fda_adapter.zip`.

In [ ]:
# 1. Is there a GPU? (If this fails: Settings > Accelerator > GPU T4 x2, then restart the session.)
!nvidia-smi
import torch; print('cuda available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. Install the same library versions the script was tested with (torch itself stays as Kaggle ships it).
!pip install -q "transformers==5.22.1" "peft==0.21.2" "accelerate==1.15.0" "bitsandbytes>=0.45"
import importlib.metadata as md
print({p: md.version(p) for p in ['torch','transformers','peft','accelerate','bitsandbytes']})

In [ ]:
# 3. Find the private dataset you attached (Add Input > your dataset 'fda-extraction-train').
import glob, os
hits = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
assert len(hits) == 1, f'expected exactly one train.jsonl under /kaggle/input, found {hits}'
DATA = os.path.dirname(hits[0]); print('data dir:', DATA, os.listdir(DATA))
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # downloaded from Hugging Face (needs Internet = On)
os.makedirs('/kaggle/working', exist_ok=True)

In [ ]:
# 4. Quick check (about 3 minutes): 16 examples, 2 optimizer steps. The loss must be a normal number, not nan.
!cd {DATA} && CUDA_VISIBLE_DEVICES=0 python train.py --model {MODEL} --data {DATA} --out /kaggle/working/smoke --max-examples 16 --accum 8 --quant 4bit --device cuda --dtype fp16 2>&1 | grep --line-buffered -v Warning | tail -25

In [ ]:
# 5. The real run. 1 epoch over 1,200 labels (150 optimizer steps). Stops and saves by itself after 6,000 s of training.
# A non-finite loss aborts it. The log is also saved to /kaggle/working/train_log.txt.
!cd {DATA} && CUDA_VISIBLE_DEVICES=0 python train.py --model {MODEL} --data {DATA} --out /kaggle/working/adapter --quant 4bit --device cuda --dtype fp16 --time-budget 6000 2>&1 | grep --line-buffered -v Warning | tee /kaggle/working/train_log.txt

In [ ]:
# 6. Zip the adapter (about 70 MB) for download, and show the final numbers.
import shutil, json
shutil.rmtree('/kaggle/working/smoke', ignore_errors=True)
shutil.make_archive('/kaggle/working/fda_adapter', 'zip', '/kaggle/working/adapter')
print(json.dumps(json.load(open('/kaggle/working/adapter/train_metrics.json')), indent=2))
print(os.listdir('/kaggle/working'))